In [ ]:
# Instantiate the estimator directly rather than nesting it inside a dictionary
tuned_model = XGBClassifier(
    n_estimators=250,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.6,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

new_pipeline = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42, sampling_strategy=0.6)),
        ('model', tuned_model)
        ])


# Define training pipeline incorporating SMOTE to balance class representations cleanly


new_pipeline.fit(X_train, y_train)

# Predict probabilities
y_probs = new_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
# Search for a decision threshold that maximizes accuracy while satisfying Recall >= 75%
best_threshold = 0.5
best_accuracy = 0.0
best_recall = 0.0

for threshold in np.arange(0.3, 0.6, 0.01):
    y_pred = (y_probs>= threshold).astype(int)
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    # Check if this threshold satisfies our targets
    if rec >= 0.75 and acc >= best_accuracy:
        best_accuracy = acc
        best_recall = rec
        best_threshold = threshold

# Apply the discovered optimal decision threshold
y_pred_final = (y_probs >= best_threshold).astype(int)
acc = accuracy_score(y_test, y_pred_final)
rec = recall_score(y_test, y_pred_final)
roc_auc= roc_auc_score(y_test, y_probs)



print(f"\nTarget Met! Selected Optimal Threshold: {best_threshold:.2f}\n")

print(f"\nModel: {name}")
print(f"Accuracy: {acc}")
print(f"Recall: {rec}")
print(f"ROC_AUC: {roc_auc}\n")
print(classification_report(y_test, y_pred_final))

cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='coolwarm')
plt.title(f"Confusion Matrix - {name}")
plt.show()

results.append((name, acc))